In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl

In [2]:
df=pd.read_csv("..//Data/bank.csv")

With the success with the simple numbered dataset, wondering if there is anyway to build another model in a different way.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [4]:
df[df['amount']==0].head(16)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
2736447,212,CASH_OUT,0.0,C1510987794,0.0,0.0,C1696624817,0.00,0.00,1,0
3247298,250,CASH_OUT,0.0,C521393327,0.0,0.0,C480398193,0.00,0.00,1,0
3760289,279,CASH_OUT,0.0,C539112012,0.0,0.0,C1106468520,538547.63,538547.63,1,0
5563714,387,CASH_OUT,0.0,C1294472700,0.0,0.0,C1325541393,7970766.57,7970766.57,1,0
5996408,425,CASH_OUT,0.0,C832555372,0.0,0.0,C1462759334,76759.90,76759.90,1,0
5996410,425,CASH_OUT,0.0,C69493310,0.0,0.0,C719711728,2921531.34,2921531.34,1,0
6168500,554,CASH_OUT,0.0,C10965156,0.0,0.0,C1493336195,230289.66,230289.66,1,0
6205440,586,CASH_OUT,0.0,C1303719003,0.0,0.0,C900608348,1328472.86,1328472.86,1,0
6266414,617,CASH_OUT,0.0,C1971175979,0.0,0.0,C1352345416,0.00,0.00,1,0
6281483,646,CASH_OUT,0.0,C2060908932,0.0,0.0,C1587892888,0.00,0.00,1,0


In [5]:

ratio = np.where(
    (df['oldbalanceOrg'] == 0) & (df['amount'] == 0),
    1,  # your custom condition
    np.where(
        df['amount'] == 0,
        np.nan,  # avoid division by zero
        (df['oldbalanceOrg'] / df['amount']) * 100
    )
)

ratio = pd.Series(ratio).round(2)

df['transfer_percent'] = pd.qcut(
    ratio,
    q=10,
    duplicates='drop'
)

I bined the transactions base on there Original balance to transfer amount ratio.


In [6]:

ratio = np.where(
    df['amount'] == 0,
    1,  # your condition
    (df['oldbalanceDest'] / df['amount']) * 100
)

ratio = pd.Series(ratio, index=df.index).round(2)

df['amount<Dest'] = pd.qcut(
    ratio,
    q=10,
    duplicates='drop'
)

I bined the transactions base on there Old Destination balance to transfer amount ratio.


In [7]:
df['amount<Dest'].isnull().sum()

np.int64(0)

making sure there is no nan values 

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 13 columns):
 #   Column            Dtype   
---  ------            -----   
 0   step              int64   
 1   type              str     
 2   amount            float64 
 3   nameOrig          str     
 4   oldbalanceOrg     float64 
 5   newbalanceOrig    float64 
 6   nameDest          str     
 7   oldbalanceDest    float64 
 8   newbalanceDest    float64 
 9   isFraud           int64   
 10  isFlaggedFraud    int64   
 11  transfer_percent  category
 12  amount<Dest       category
dtypes: category(2), float64(5), int64(3), str(3)
memory usage: 546.1 MB


In [9]:
df=df.drop(columns=['step','nameOrig','nameDest','isFlaggedFraud'])

removing ['step','nameOrig','nameDest','isFlaggedFraud']

In [10]:
df=pd.get_dummies(data=df,columns=['type','transfer_percent','amount<Dest'])

dummy all the categorical columns ['type','transfer_percent','amount<Dest']

In [11]:
df.to_csv('../Data/bank_cleaned_2.csv', index=False)